# Generate ESM2 Embeddings for DiffDock Evaluation

## Why this is needed

DiffDock's score model uses **ESM2 language model embeddings** as features for each residue in the receptor protein.
These are pre-computed vectors that encode the biochemical identity of each amino acid in context — essentially
a learned representation of what each residue 'means' biologically, based on patterns in millions of protein sequences.

When you run **`inference.py`** on a single protein, these embeddings are generated automatically on the fly.
But when you run **`evaluate.py`** on a whole dataset (e.g. PDBBind), they must be pre-computed and saved to a
`.pt` file first — otherwise every complex would recompute them, which is very slow.

## What this notebook does

1. Reads all protein PDB files in a dataset directory
2. Extracts the amino acid sequences for each chain
3. Runs ESM2-650M on all sequences to produce per-residue embedding vectors (shape: `seq_len × 1280`)
4. Saves a single `.pt` file mapping `{pdb_id}_chain_{i}` → embedding tensor

The output file is then passed to `evaluate.py` via `--esm_embeddings_path`.

## When to re-run

Re-run whenever you evaluate on a new dataset directory or split that includes proteins not covered
by an existing embeddings file. You can safely run on a superset of complexes — the evaluation script
only loads embeddings for the complexes in its split.

In [ ]:
import sys
from pathlib import Path

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import os
import torch
from tqdm import tqdm
from Bio.PDB import PDBParser
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

from datasets.constants import three_to_one
from utils.inference_utils import compute_ESM_embeddings

print('Project root:', project_root)
print('CUDA available:', torch.cuda.is_available())

## Configuration

Set `DATA_DIR` to the folder containing your PDBBind-formatted complexes (one subfolder per complex,
each containing a `{name}_protein_processed.pdb` or `{name}_protein.pdb` file).

Set `OUTPUT_PATH` to where the resulting `.pt` file should be saved.
By convention this lives in `data/` and is passed as `--esm_embeddings_path` to `evaluate.py`.

Examples:
```
DATA_DIR = '/home/qf226/rds/hpc-work/data/PDBBind_processed'   # 8-complex test set
DATA_DIR = '/home/qf226/rds/hpc-work/data/PDBBind_processed'         # full PDBBind set
```

In [ ]:
# ── Configure these paths ──────────────────────────────────────────────────────
DATA_DIR    = project_root / 'data' / 'PDBBind_processed_test'
OUTPUT_PATH = project_root / 'data' / 'esm2_3billion_embeddings.pt'
# ──────────────────────────────────────────────────────────────────────────────

assert DATA_DIR.exists(), f'DATA_DIR not found: {DATA_DIR}'
print(f'Dataset:  {DATA_DIR}')
print(f'Output:   {OUTPUT_PATH}')

## Step 1 — Extract amino acid sequences from PDB files

For each complex, we parse the processed protein PDB and extract the one-letter amino acid sequence
for every chain. Only residues with Cα, N, and C backbone atoms are kept (i.e. proper amino acids;
water molecules and ligands are skipped). Unknown residue names are replaced with `-`.

The result is a list of `(label, sequence)` pairs where label is `{pdb_id}_chain_{i}`.
This matches the key format expected by `evaluate.py`.

In [ ]:
biopython_parser = PDBParser(QUIET=True)

def extract_sequences(data_dir: Path) -> tuple[list[str], list[str]]:
    """Extract per-chain amino acid sequences from all complexes in data_dir.

    Args:
        data_dir: Directory containing one subfolder per complex.

    Returns:
        labels:    List of chain identifiers, e.g. ['6d08_chain_0', '6d08_chain_1', ...].
        sequences: Corresponding one-letter amino acid sequences.
    """
    labels, sequences = [], []
    complex_names = sorted(p.name for p in data_dir.iterdir() if p.is_dir())

    for name in tqdm(complex_names, desc='Extracting sequences'):
        # Prefer the pre-processed file; fall back to raw protein file.
        pdb_path = data_dir / name / f'{name}_protein_processed.pdb'
        if not pdb_path.exists():
            pdb_path = data_dir / name / f'{name}_protein.pdb'
        if not pdb_path.exists():
            print(f'  WARNING: no protein PDB found for {name}, skipping')
            continue

        structure = biopython_parser.get_structure('x', pdb_path)[0]
        for chain_idx, chain in enumerate(structure):
            seq = ''
            for residue in chain:
                if residue.get_resname() == 'HOH':
                    continue
                has_backbone = all(residue.has_id(a) for a in ('CA', 'N', 'C'))
                if not has_backbone:
                    continue
                seq += three_to_one.get(residue.get_resname(), '-')
            if seq:
                labels.append(f'{name}_chain_{chain_idx}')
                sequences.append(seq)

    return labels, sequences


labels, sequences = extract_sequences(DATA_DIR)
print(f'\nFound {len(labels)} chains across {len(set(l.split("_chain_")[0] for l in labels))} complexes')
for label, seq in zip(labels, sequences):
    print(f'  {label}: {len(seq)} residues')

## Step 2 — Compute ESM2 embeddings

We use **ESM2-650M** (`esm2_t33_650M_UR50D`) — 33 transformer layers, hidden dim 1280 — which is the
same model used by `inference.py` when computing embeddings on the fly.

The model automatically moves to GPU if available. For 8 proteins this takes ~10 seconds on GPU
or a few minutes on CPU. For the full PDBBind (~4000 complexes) you should run this on a GPU node.

Each embedding has shape `(seq_len, 1280)` — one 1280-dimensional vector per residue.

In [ ]:
import esm

print('Loading ESM2-650M...')
esm_model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
esm_model.eval()
if torch.cuda.is_available():
    esm_model = esm_model.cuda()
    print('Model on GPU')
else:
    print('Model on CPU (will be slow for large datasets — consider running on a GPU node)')

print('\nComputing embeddings...')
embeddings = compute_ESM_embeddings(esm_model, alphabet, labels, sequences)

print(f'\nGenerated embeddings for {len(embeddings)} chains')
sample_key = next(iter(embeddings))
print(f'Example — {sample_key}: shape {embeddings[sample_key].shape}  (seq_len × embed_dim)')

## Step 3 — Save to `.pt` file

The embeddings are saved as a single PyTorch dict: `{pdb_id}_chain_{i}` → tensor.
This is the format expected by `evaluate.py` (and `pdbbind.py` underneath it).

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(embeddings, OUTPUT_PATH)
size_mb = OUTPUT_PATH.stat().st_size / 1e6
print(f'Saved {len(embeddings)} embeddings to {OUTPUT_PATH}  ({size_mb:.1f} MB)')

## Step 4 — Verify

Quick sanity check: reload the file and confirm the keys and shapes look right.

In [ ]:
loaded = torch.load(OUTPUT_PATH)
print(f'Keys in file: {len(loaded)}')
for key, tensor in sorted(loaded.items()):
    print(f'  {key}: {tuple(tensor.shape)}')

## Running evaluation with the embeddings

Pass `--esm_embeddings_path` to `evaluate.py`:

```bash
source ~/.bashrc && conda activate diffdock
export LD_LIBRARY_PATH="$CONDA_PREFIX/lib:$LD_LIBRARY_PATH"

python evaluate.py \
    --model_dir workdir/v1.1/score_model \
    --confidence_model_dir workdir/v1.1/confidence_model \
    --dataset pdbbind \
    --data_dir /home/qf226/rds/hpc-work/data/PDBBind_processed \
    --split_path data/splits/test_small_split.txt \
    --esm_embeddings_path /home/qf226/rds/hpc-work/data/embeddings/pdbbind_esm2.pt \
    --cache_path /home/qf226/rds/hpc-work/data/cache_torsion \
    --out_dir results/eval_small_test \
    --samples_per_complex 40 \
    --batch_size 40 \
    --inference_steps 20 \
    --no_final_step_noise \
    --save_predictions \
    --num_workers 4
```

**Note on caching:** `evaluate.py` caches the processed graph dataset in `/home/qf226/rds/hpc-work/data/cache_torsion/`. The cache key
includes whether ESM embeddings were used (`_esmEmbeddings` suffix). If you previously ran evaluate without
embeddings and now add them, a new cache entry will be built automatically — you don't need to delete anything.